## Adaptive pretraining

### Colab Setup

In [1]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Key Imports

In [2]:
import pandas as pd
import torch

from config import APT, APT_EPOCHS, IDIOMS, RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.apt_pools import build_pools, build_tapt_pool
from data.loader_twd_labelled import load_splits
from models.apt import adapt
from models.frozen_probe import probe
from models.plm_finetune import finetune
from sklearn.metrics import f1_score

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Arms

In [ ]:
VANILLA = SHAH_PLM[ENC]["model_name"]
FOMC_POOL = build_pools(verbose=False)[0]["sentence"].to_list()
print(f"fomc pool: {len(FOMC_POOL):,} sentences")

# arm -> (sentences, epochs key, starting checkpoint, seed-dependent)
ARMS = {
    "dapt-fomc": (lambda seed: FOMC_POOL, "dapt", VANILLA, False),
    "tapt": (
        lambda seed: load_splits("benchmark", seed=seed)[0]["sentence"].to_list(),
        "tapt",
        VANILLA,
        True,
    ),
    "curated-tapt": (
        lambda seed: build_tapt_pool(seed)["sentence"].to_list(),
        "curated-tapt",
        VANILLA,
        True,
    ),
    "dapt-fomc+curated-tapt": (
        lambda seed: build_tapt_pool(seed)["sentence"].to_list(),
        "curated-tapt",
        str(RESULTS_DIR / "models" / "dapt-fomc"),
        True,
    ),
}


### Continued pretraining

In [4]:
for arm, (pool_fn, epochs_key, start, per_seed) in ARMS.items():
    for seed in SEEDS if per_seed else [None]:
        name = f"{arm}-s{seed}" if per_seed else arm
        save_dir = str(RESULTS_DIR / "models" / name)
        if os.path.isdir(save_dir):
            print(f"{name}: already adapted, skipping")
            continue
        sentences = pool_fn(seed)
        print(f"{name}: {len(sentences):,} sentences", flush=True)
        adapt(
            sentences,
            model_name=start,
            epochs=APT_EPOCHS[epochs_key],
            save_dir=save_dir,
            device=DEVICE,
            verbose=True,
            **APT,
        )

downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmpyv0fsg48
  meeting_minutes: 230 docs -> 47,340 sentences
  speech: 1026 docs -> 107,548 sentences
  press_conference: 63 docs -> 24,750 sentences


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
dapt-fomc: 164,688 sentences


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 164,688 sentences...
    training: 5,147 batches/epoch x 1 epoch(s), 5,147 updates at effective batch 32
    100/5,147: mlm loss 1.4254 | 2.7 it/s, eta 31 min
    200/5,147: mlm loss 1.4018 | 2.7 it/s, eta 30 min
    300/5,147: mlm loss 1.3937 | 2.7 it/s, eta 30 min
    400/5,147: mlm loss 1.3861 | 2.8 it/s, eta 29 min
    500/5,147: mlm loss 1.3760 | 2.8 it/s, eta 28 min
    600/5,147: mlm loss 1.3713 | 2.8 it/s, eta 27 min
    700/5,147: mlm loss 1.3727 | 2.8 it/s, eta 27 min
    800/5,147: mlm loss 1.3680 | 2.8 it/s, eta 26 min
    900/5,147: mlm loss 1.3645 | 2.8 it/s, eta 25 min
    1,000/5,147: mlm loss 1.3604 | 2.8 it/s, eta 25 min
    1,100/5,147: mlm loss 1.3571 | 2.8 it/s, eta 24 min
    1,200/5,147: mlm loss 1.3522 | 2.8 it/s, eta 24 min
    1,300/5,147: mlm loss 1.3453 | 2.8 it/s, eta 23 min
    1,400/5,147: mlm loss 1.3377 | 2.8 it/s, eta 22 min
    1,500/5,147: mlm loss 1.3324 | 2.8 it/s, eta 22 min
    1,600/5,147: mlm loss 1.3261 | 2.8 it/s, eta 21 min
  

...

    saved -> /content/drive/MyDrive/thesis/models/dapt-fomc
downloading TDW repo tarball (~61MB)...
extracted -> /tmp/tmp26ffbsk7
  meeting_minutes: 230 docs -> 47,340 sentences
  speech: 1026 docs -> 107,548 sentences
  press_conference: 63 docs -> 24,750 sentences
Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
dapt-global: 439,414 sentences


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 439,414 sentences...
    training: 13,732 batches/epoch x 1 epoch(s), 13,732 updates at effective batch 32
    100/13,732: mlm loss 1.6616 | 2.7 it/s, eta 86 min
    200/13,732: mlm loss 1.6266 | 2.5 it/s, eta 89 min
    300/13,732: mlm loss 1.5819 | 2.5 it/s, eta 88 min
    400/13,732: mlm loss 1.5516 | 2.5 it/s, eta 87 min
    500/13,732: mlm loss 1.5400 | 2.6 it/s, eta 86 min
    600/13,732: mlm loss 1.5212 | 2.6 it/s, eta 85 min
    700/13,732: mlm loss 1.5113 | 2.6 it/s, eta 85 min
    800/13,732: mlm loss 1.4969 | 2.6 it/s, eta 84 min
    900/13,732: mlm loss 1.4892 | 2.6 it/s, eta 84 min
    1,000/13,732: mlm loss 1.4758 | 2.6 it/s, eta 83 min
    1,100/13,732: mlm loss 1.4691 | 2.6 it/s, eta 82 min
    1,200/13,732: mlm loss 1.4634 | 2.6 it/s, eta 81 min
    1,300/13,732: mlm loss 1.4549 | 2.6 it/s, eta 81 min
    1,400/13,732: mlm loss 1.4442 | 2.6 it/s, eta 80 min
    1,500/13,732: mlm loss 1.4376 | 2.6 it/s, eta 79 min
    1,600/13,732: mlm loss 1.4330 | 2.6 i

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved -> /content/drive/MyDrive/thesis/models/dapt-global


### Held-out MLM loss

In [ ]:
# decontamination removes the labelled benchmark from the pools, so dapt-fomc has
# never seen these 1,984 train sentences. DAPT only: the TAPT arms pretrain on
# exactly this text, and the idiom probe covers them instead. test is untouched.
eval_on, _ = load_splits("benchmark", seed=SEEDS[0])
eval_on = eval_on["sentence"].to_list()

from torch.utils.data import DataLoader
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
)



def mlm_loss(model_path, seed=0):
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMaskedLM.from_pretrained(model_path).to(DEVICE).eval()
    enc = tok(eval_on, truncation=True, max_length=APT["max_len"])
    rows = [
        {"input_ids": i, "attention_mask": m}
        for i, m in zip(enc["input_ids"], enc["attention_mask"])
    ]
    dl = DataLoader(
        rows,
        batch_size=APT["batch_size"],
        collate_fn=DataCollatorForLanguageModeling(
            tok, mlm_probability=APT["mlm_probability"]
        ),
    )
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            total += model(**batch).loss.item()
            n += 1
    del model
    torch.cuda.empty_cache()
    return total / n


paths = {ENC: VANILLA, "dapt-fomc": str(RESULTS_DIR / "models" / "dapt-fomc")}

rows = [dict(model=k, mlm_loss=round(mlm_loss(v), 4)) for k, v in paths.items()]
for r in rows:
    print(f"{r['model']}: {r['mlm_loss']}")

pd.DataFrame(rows).to_csv(RESULTS_DIR / "mlm_loss.csv", index=False)
print("saved ->", RESULTS_DIR / "mlm_loss.csv")


### Masked-idiom probe

In [ ]:
from transformers import pipeline


def probe_idioms(model_path):
    mlm = pipeline("fill-mask", model=model_path, device=0 if DEVICE == "cuda" else -1)
    mask = mlm.tokenizer.mask_token
    rows = []
    for phrase, gold in IDIOMS:
        top5 = [
            r["token_str"].strip().lower()
            for r in mlm(phrase.replace("[MASK]", mask), top_k=5)
        ]
        rows.append(
            dict(phrase=phrase, gold=gold, hit=gold.lower() in top5, top5="|".join(top5))
        )
    del mlm
    torch.cuda.empty_cache()
    return rows


models = {ENC: VANILLA}
for arm, (_, _, _, per_seed) in ARMS.items():
    name = f"{arm}-s{SEEDS[0]}" if per_seed else arm
    models[arm] = str(RESULTS_DIR / "models" / name)

records = []
for label, path in models.items():
    rows = probe_idioms(path)
    records += [dict(model=label, **r) for r in rows]
    print(f"{label}: {sum(r['hit'] for r in rows)}/{len(rows)}", flush=True)

idf = pd.DataFrame(records)
idf.to_csv(RESULTS_DIR / "idioms.csv", index=False)
print("saved ->", RESULTS_DIR / "idioms.csv")

SyntaxError: invalid syntax (1975882075.py, line 11)

### Fine-tune

In [ ]:
cfg = SHAH_PLM[ENC]

for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=str(RESULTS_DIR / "models" / name),
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{model_key} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()

### Frozen probe

In [ ]:
for arm, (_, _, _, per_seed) in ARMS.items():
    for seed in SEEDS:
        model_key = f"frozen-{arm}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        name = f"{arm}-s{seed}" if per_seed else arm
        train, test = load_splits("benchmark", seed=seed)
        pred = probe(
            train,
            test,
            model_name=str(RESULTS_DIR / "models" / name),
            device=DEVICE,
            seed=seed,
        )
        true = test["label"].to_list()
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs="",
            weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
            macro_f1=round(f1_score(true, pred, average="macro"), 4),
        )
        print(f"{model_key} seed {seed}: macro={f1_score(true, pred, average='macro'):.4f}")
